                # BootstrapFewShotWithRandomSearch (BootstrapRS)

                Build several bootstrapped demo sets and choose among them using the frozen validation split.

                **Use it when:** BootstrapFewShot is promising and you can afford multiple candidate programs to reduce dependence on one demo sample.

                **What compilation changes:** Demonstrations only, but candidate selection adds a validation-driven search loop.

                Important configuration in this benchmark:

                - 8 candidate programs in the full profile (2 in smoke)
- two bootstrapped plus two labeled demos per candidate
- single evaluation thread by default

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'bootstrap-random-search'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='bootstrap-random-search'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BootstrapFewShotWithRandomSearch(
    metric=exact_match, num_candidate_programs=profile.bootstrap_candidates,
    max_bootstrapped_demos=2, max_labeled_demos=2, num_threads=1,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, valset=valset)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BootstrapRS — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 70.0%
uplift: +20.0 points vs Luna reference
optimization: $0.2860 and 6.6min
inference latency: mean 1.79s; p95 2.66s
reload checks: prompt=True; model=None; predictions=3/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/bootstrap-random-search.json
- canonical prompt: chapter06/results/final_prompts/bootstrap-random-search.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

The extra compile spend buys candidate selection, not a new instruction. Check whether the held-out gain justifies the search relative to plain BootstrapFewShot.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 2
1. is_ai=False: We've covered the "CR" of CRUD. Now let's move on to the "U" (Update). Updating a resource is very similar to creating a resource. They are both multi-step processes. First, the…
2. is_ai=True: Having covered the “CR” in CRUD, we now turn to “U” for Update. Updating a resource closely resembles creating one, for both are multi-step affairs. First, the user requests a f…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.